# Feature Engineering

This notebook transforms the preprocessed mental health text into numerical features that can be used by machine learning classification algorithms. Term Frequency-Inverse Document Frequency (TF-IDF) is applied to represent each social media post based on the importance of its words within the dataset.

The feature engineering process includes separating the text and target labels, dividing the dataset into training and testing sets, fitting the TF-IDF vectorizer on the training data, and transforming both datasets into numerical feature matrices.

## 1. Import Libraries

The required libraries are imported for data manipulation, dataset splitting, TF-IDF vectorization, and saving the fitted vectorizer for later use.

In [125]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

## 2. Load and Validate the Preprocessed Dataset

The final preprocessed dataset is loaded from the processed data folder. Basic validation is performed to confirm the dataset dimensions, column names, missing values, and class distribution before creating machine learning features.

In [126]:
# Define the path to the preprocessed dataset
data_path = Path("../data/processed/mental_health_text_preprocessed.csv")

# Load the dataset
df = pd.read_csv(data_path)

# Display the first five records
df.head()

,Label,TEXT,character_count,word_count,TEXT_ORIGINAL,TEXT_REPAIRED,TEXT_NORMALIZED,TEXT_PROCESSED
0,0.0,TIL the movie Starship Troopers was never adap...,89,14,TIL the movie Starship Troopers was never adap...,TIL the movie Starship Troopers was never adap...,TIL the movie Starship Troopers was never adap...,til movie starship troopers never adapt succes...
1,0.0,What do you call a fat baby?,28,7,What do you call a fat baby?,What do you call a fat baby?,What do you call a fat baby?,fat baby
2,0.0,Two morons are sitting on a fence. The big one...,78,16,Two morons are sitting on a fence. The big one...,Two morons are sitting on a fence. The big one...,Two morons are sitting on a fence. The big one...,number moron sit fence big number fall not
3,0.0,I covered all my weapons in glue.,33,7,I covered all my weapons in glue.,I covered all my weapons in glue.,I covered all my weapons in glue.,cover weapon glue
4,0.0,Joke I made up: Caveman and a bear walk into a...,103,19,Joke I made up: Caveman and a bear walk into a...,Joke I made up: Caveman and a bear walk into a...,Joke I made up: Caveman and a bear walk into a...,joke caveman bear walk bar ba ender say story ...


In [127]:
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print("\nMissing values:")
print(df.isna().sum())

print("\nClass distribution:")
print(df["Label"].value_counts().sort_index())

Dataset shape: (15774, 8)

Columns: ['Label', 'TEXT', 'character_count', 'word_count', 'TEXT_ORIGINAL', 'TEXT_REPAIRED', 'TEXT_NORMALIZED', 'TEXT_PROCESSED']

Missing values:
Label              0
TEXT               0
character_count    0
word_count         0
TEXT_ORIGINAL      0
TEXT_REPAIRED      0
TEXT_NORMALIZED    0
TEXT_PROCESSED     0
dtype: int64

Class distribution:
Label
0.0    6032
1.0    4296
2.0    5446
Name: count, dtype: int64


## 3. Separate Features and Target Labels

The dataset is divided into input features (`X`) and target labels (`y`). The `TEXT` column contains the preprocessed social media posts that will be transformed into numerical features, while the `Label` column contains the corresponding classification labels.

In [128]:
# Define the predictor and target variables
X = df["TEXT_PROCESSED"]
y = df["Label"]

print(f"Number of text samples: {len(X):,}")
print(f"Number of labels: {len(y):,}")

print("\nTarget class distribution:")
print(y.value_counts().sort_index())

Number of text samples: 15,774
Number of labels: 15,774

Target class distribution:
Label
0.0    6032
1.0    4296
2.0    5446
Name: count, dtype: int64


## 4. Split the Dataset into Training and Testing Sets

The dataset is divided into separate training and testing subsets before feature engineering. The training data are used to fit the TF-IDF vectorizer and train the machine learning models, while the testing data remain unseen during training and are used to evaluate model performance.

An 80/20 split is used, with stratified sampling to preserve the original class distribution in both subsets. Performing the split before fitting the TF-IDF vectorizer helps prevent data leakage by ensuring that information from the testing data does not influence feature generation.

In [129]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"Training samples: {len(X_train):,}")
print(f"Testing samples: {len(X_test):,}")

print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nTesting class distribution:")
print(y_test.value_counts(normalize=True).sort_index())

Training samples: 12,619
Testing samples: 3,155

Training class distribution:
Label
0.0    0.382360
1.0    0.272367
2.0    0.345273
Name: proportion, dtype: float64

Testing class distribution:
Label
0.0    0.382567
1.0    0.272266
2.0    0.345166
Name: proportion, dtype: float64


## 5. Apply TF-IDF Vectorization

Machine learning algorithms require numerical input rather than raw text. Term Frequency-Inverse Document Frequency (TF-IDF) is used to convert the preprocessed text into numerical feature vectors that represent the relative importance of words within each document.

The TF-IDF vectorizer is fit using only the training data to prevent data leakage. The learned vocabulary is then applied to both the training and testing datasets to create consistent feature representations.

In [130]:
# Create the TF-IDF vectorizer
tfidf = TfidfVectorizer()

# Learn the vocabulary from the training data
X_train_tfidf = tfidf.fit_transform(X_train)

# Apply the learned vocabulary to the testing data
X_test_tfidf = tfidf.transform(X_test)

In [131]:
print(f"Training feature matrix shape: {X_train_tfidf.shape}")
print(f"Testing feature matrix shape: {X_test_tfidf.shape}")

print(f"\nVocabulary size: {len(tfidf.vocabulary_):,} unique terms")
print(type(X_train_tfidf))

Training feature matrix shape: (12619, 23837)
Testing feature matrix shape: (3155, 23837)

Vocabulary size: 23,837 unique terms
<class 'scipy.sparse._csr.csr_matrix'>


## 6. Examine the TF-IDF Feature Matrix

The TF-IDF vectorizer converts the text into a high-dimensional numerical feature matrix. Each feature corresponds to a unique term learned from the training data, and each value represents the importance of that term within a document relative to the entire corpus.

The following examples illustrate the size of the feature matrix and a sample of the learned vocabulary.

In [132]:
# Display the highest-weighted TF-IDF terms for the first training document

doc = 0

row = X_train_tfidf[doc].toarray().flatten()

nonzero = row.nonzero()[0]

weights = pd.DataFrame(
    {
        "Term": feature_names[nonzero],
        "TF-IDF Weight": row[nonzero],
    }
)

weights.sort_values(
    by="TF-IDF Weight",
    ascending=False,
).head(15)

,Term,TF-IDF Weight
17,dacha,0.420621
28,healer,0.296730
16,cyborgs,0.264923
20,dms,0.205771
35,objected,0.190562
23,enlightened,0.190546
8,awaiting,0.181904
43,recite,0.180407
19,discarded,0.173263
38,pirated,0.171661


In [133]:
X_train.iloc[0]

'family collapse childhood mother want leave father number year old father ask stay wait grow grow yesterday witness treason eye father go business trip leave mother action mother not long come call friend work buy alcohol arrange feast house come home school tired go bed woke noise number feast continue pa icipation ce ain man not know stay number call taxi mother go'

In [134]:
print(X.name)
print(X_train.iloc[0])

TEXT_PROCESSED
family collapse childhood mother want leave father number year old father ask stay wait grow grow yesterday witness treason eye father go business trip leave mother action mother not long come call friend work buy alcohol arrange feast house come home school tired go bed woke noise number feast continue pa icipation ce ain man not know stay number call taxi mother go


## 7. Feature Engineering Summary

The TF-IDF vectorizer successfully transformed the preprocessed text into a sparse numerical feature matrix suitable for machine learning. The vectorizer learned its vocabulary from the training dataset and applied the same feature representation to both the training and testing datasets, preventing data leakage. These numerical features will serve as the input for the machine learning classification models developed in the next notebook.

In [135]:
print("Feature Engineering Summary")
print("-" * 35)

print(f"Training documents : {X_train_tfidf.shape[0]:,}")
print(f"Testing documents  : {X_test_tfidf.shape[0]:,}")
print(f"Vocabulary size    : {len(tfidf.vocabulary_):,}")
print(f"Feature type       : {type(X_train_tfidf).__name__}")

Feature Engineering Summary
-----------------------------------
Training documents : 12,619
Testing documents  : 3,155
Vocabulary size    : 23,837
Feature type       : csr_matrix


Saving the fitted vectorizer preserves the learned vocabulary so that future datasets can be transformed using the same feature representation.

In [136]:
from pathlib import Path

# Create the models directory if it doesn't already exist
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

# Save the vectorizer
joblib.dump(tfidf, models_dir / "tfidf_vectorizer.joblib")

print(f"Vectorizer saved to: {models_dir / 'tfidf_vectorizer.joblib'}")

Vectorizer saved to: ..\models\tfidf_vectorizer.joblib


The feature engineering phase successfully transformed the preprocessed text into sparse numerical representations suitable for machine learning. These features will be used in the next notebook to train, evaluate, and compare multiple text classification models.